# PointNet++ v6.0.0 — Standard (PointNet++ + Triplet) — Train + Eval

**Hipotesis utama v0.6.0**: ArcFace loss memberikan keuntungan signifikan dibandingkan Triplet batch-hard loss pada 3D palm identification berbasis PointNet++ dalam regime low-data (1 sampel/sesi).

**Notebook ini melatih varian**: `standard` (Triplet batch-hard loss, margin=0.3).

**Setup (carry-over dari v5.0.0)**:
- **PointNet++ murni** (no GeoAtt, no aux classifier, no geometry features) — fair comparison hanya berbeda di loss
- 10 subjek (gede di-drop, <15 sesi valid)
- Split deterministic chronological 8 train / 2 val / 2 test / 3 holdout per subjek
- 1 median frame per sesi (low-data regime)
- Val pair EER untuk model selection (smoothed window=5)
- Fixed budget: 120 + 30 epoch, no early stopping
- Augmentation depth-focused: rotation ±45°, tilt ±20°, XYZ translation ±3cm, scale 0.95–1.05, jitter σ=1mm
- 10 seeds: 42, 123, 2024, 7, 31337, 0, 1, 2, 3, 4

**v6.0.0 patch (sudah landed)**: `train.py` sudah mendukung `--loss arcface` di low-data regime (PalmFrameDataset, num_classes=n_subjects, head ArcFace, dispatcher per-frame `_run_perframe_epoch`). Flag baru: `--arcface-margin`, `--arcface-scale`. Tidak butuh F2.0 strip GeoAtt (cukup default `no_geom` + tanpa `--use-aux-loss`; modul GeoAtt dormant tapi tidak diaktifkan).

## 1. Setup — GitHub Clone + Dataset in Repo

**Prerequisite**: simpan GitHub Personal Access Token sebagai Colab Secret:
1. Sidebar kiri → 🔑 Secrets → Add Secret
2. Name: `GITHUB_TOKEN`, Value: token dari https://github.com/settings/tokens
3. Toggle "Notebook access" ON

Dataset (`3DCNN/dataset/`) sudah include di repo — tidak perlu Google Drive mount lagi.

In [ ]:
from google.colab import userdata
import os, sys, subprocess
from pathlib import Path

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/RZulfikri/Thesis.git'
REPO_DIR = Path('/content/Thesis')
PROJECT_ROOT = REPO_DIR / '3DCNN'
COLAB_BRANCH = 'colab'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin

os.chdir(str(REPO_DIR))
ret = subprocess.run(['git', 'checkout', COLAB_BRANCH], capture_output=True)
if ret.returncode != 0:
    !git checkout -b {COLAB_BRANCH}
    print(f'Created new branch: {COLAB_BRANCH}')
else:
    !git merge origin/main --no-edit --strategy-option theirs 2>/dev/null || true
    print(f'Checked out existing branch: {COLAB_BRANCH}')

!git config user.email "colab-runner@thesis.local"
!git config user.name "Colab Runner"

DATASET_DST = PROJECT_ROOT / 'dataset'
if DATASET_DST.exists():
    print(f'Dataset ditemukan di repo: {DATASET_DST}')
else:
    print(f'⚠️ Dataset tidak ditemukan di repo: {DATASET_DST}')

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(str(PROJECT_ROOT))

!ls -la | head -20
print(f'\nPROJECT_ROOT : {PROJECT_ROOT}')
print(f'Branch       : {COLAB_BRANCH}')
print(f'Dataset      : {DATASET_DST}')


In [ ]:
!python -c "from utils.dataset_lowdata import DROPPED_SUBJECTS; print('DROPPED:', DROPPED_SUBJECTS)"
!find . -name __pycache__ -type d -exec rm -rf {} + 2>/dev/null
print('✅ Setup OK')


## 2. Launch TensorBoard

In [ ]:
import os
os.environ['TENSORBOARD_BINARY'] = '/usr/local/bin/tensorboard'

%load_ext tensorboard
%tensorboard --logdir /content/Thesis/3DCNN/runs/v6_lowdata

## 3. Configuration + GPU Probe

In [ ]:
import subprocess, os, sys, gc
import torch

SEEDS = [42, 123, 2024, 7, 31337, 0, 1, 2, 3, 4]
VARIANT = "standard"      # logical label untuk folder layout

DATA_DIR = PROJECT_ROOT / "dataset"
RUNS_DIR = PROJECT_ROOT / "runs" / "v6_lowdata"
EVAL_DIR = PROJECT_ROOT / "eval_results" / "v6_lowdata"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# v6.0.0 — Fixed budget identik antar varian (apple-to-apple)
EPOCHS          = 120
FINETUNE_EPOCHS = 30
TRIPLET_MARGIN  = 0.3        # baseline (no-op untuk arcface)
ARC_MARGIN      = 0.5        # ArcFace m  (default model)
ARC_SCALE       = 30.0       # ArcFace s  (default model)

# ============================================================
# DYNAMIC VRAM PROBE — target ~75% (conservative)
# Identik dengan v5 (apple-to-apple GPU footprint).
# ============================================================
TARGET_VRAM_FRACTION = 0.75
N_POINTS             = 16384
MAX_BS_FOR_LARGE_N   = 192 if N_POINTS > 8192 else 99999
AMP_MODE             = "bf16" if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else "fp16"
PRELOAD_AUGMENT      = True
NUM_WORKERS          = 8

try:
    out = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=memory.total,name', '--format=csv,noheader,nounits'],
        text=True
    )
    line = out.strip().split('\n')[0]
    vram_str, gpu_name = [x.strip() for x in line.split(',')]
    VRAM_GB  = int(vram_str) / 1024
    GPU_NAME = gpu_name
except Exception as e:
    print(f'GPU detection failed: {e}')
    VRAM_GB  = 0
    GPU_NAME = 'Unknown'

bf16_ok        = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
target_vram_gb = VRAM_GB * TARGET_VRAM_FRACTION

print(f'GPU            : {GPU_NAME} ({VRAM_GB:.1f} GB)')
print(f'BF16 support   : {bf16_ok}')
print(f'AMP mode       : {AMP_MODE}')
print(f'Target VRAM    : {target_vram_gb:.1f} GB ({TARGET_VRAM_FRACTION*100:.0f}% of {VRAM_GB:.0f} GB)')
print(f'N_POINTS       : {N_POINTS}')

# --- Probe BS pure PointNet++ (no_geom, no aux) ---
def probe_max_batch_size(n_points: int, target_gb: float, amp_dtype):
    sys.path.insert(0, str(PROJECT_ROOT))
    from models.siamese import SiamesePalmNet
    device = torch.device('cuda')

    def _try_bs(bs: int) -> float:
        torch.cuda.empty_cache(); gc.collect()
        torch.cuda.reset_peak_memory_stats()
        try:
            model = SiamesePalmNet(
                geom_dim=13, use_geom=False, use_aux_loss=False,
                n_subjects=10, siamese_mode="concat",
            ).to(device)
            model.train()
            opt = torch.optim.Adam(model.parameters(), lr=1e-3, fused=True)
            pts  = torch.randn(bs, n_points, 6, device=device)
            geom = torch.randn(bs, 13, device=device)
            ctx = torch.cuda.amp.autocast(dtype=amp_dtype) if amp_dtype else torch.cuda.amp.autocast(enabled=False)
            with ctx:
                emb = model.encode(pts, geom)
                loss = emb.sum()
            loss.backward(); opt.step(); torch.cuda.synchronize()
            peak_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
            del model, opt, pts, geom, emb, loss
            torch.cuda.empty_cache(); gc.collect()
            return peak_gb
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            return float('inf')
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                torch.cuda.empty_cache(); gc.collect()
                return float('inf')
            raise

    amp_dtype_local = torch.bfloat16 if AMP_MODE == 'bf16' else (torch.float16 if AMP_MODE == 'fp16' else None)
    print(f'\n  Probing max BS (target {target_gb:.1f} GB)...')
    bs_low, bs_high = 32, None
    last_peak = 0.0
    bs = 32
    while True:
        peak = _try_bs(bs)
        if peak == float('inf'):
            print(f'    BS={bs:4d} → OOM'); bs_high = bs; break
        print(f'    BS={bs:4d} → {peak:.1f} GB ({100*peak/VRAM_GB:.0f}%)')
        last_peak = peak
        if peak >= target_gb * 0.95:
            bs_high = int(bs * (target_gb / peak) * 1.1); break
        bs_low = bs; bs *= 2
    if bs_high is None: bs_high = bs_low * 2
    best_bs, best_peak = bs_low, last_peak
    while bs_high - bs_low > 16:
        bs_mid = (bs_low + bs_high) // 2
        peak = _try_bs(bs_mid)
        if peak == float('inf') or peak > VRAM_GB * 0.95:
            print(f'    BS={bs_mid:4d} → OOM/over-budget'); bs_high = bs_mid
        else:
            print(f'    BS={bs_mid:4d} → {peak:.1f} GB ({100*peak/VRAM_GB:.0f}%)')
            if peak <= target_gb:
                best_bs, best_peak = bs_mid, peak; bs_low = bs_mid
            else:
                bs_high = bs_mid
    final_bs = max(32, int(best_bs * 0.90))
    if N_POINTS > 8192:
        final_bs = min(final_bs, MAX_BS_FOR_LARGE_N)
        print(f'  [N-cap] BS dikunci ≤ {MAX_BS_FOR_LARGE_N} (N_POINTS={N_POINTS} > 8192)')
    print(f'  → Chosen BS = {final_bs} (peak ~{best_peak:.1f} GB)')
    return final_bs, best_peak

amp_dtype_main = None if AMP_MODE == 'none' else (torch.bfloat16 if AMP_MODE == 'bf16' else torch.float16)
print('\n=== PROBE (pure PointNet++ — identik antara standard/arcface) ===')
BS, peak_gb = probe_max_batch_size(N_POINTS, target_vram_gb, amp_dtype=amp_dtype_main)

def compute_repeat(bs: int, train_frames: int = 80, min_steps: int = 4) -> int:
    bs_eff = min(bs, 512)
    return max(4, -(-bs_eff * min_steps // train_frames))

REPEAT = compute_repeat(BS)

def ram_mb(bs, repeat):
    return (80 * repeat * N_POINTS * 6 * 4) / (1024 ** 2)

print('\n' + '=' * 60)
print(f'AUTO-TUNE RESULT — varian: {VARIANT}')
print('=' * 60)
print(f'  N_POINTS        : {N_POINTS}')
print(f'  AMP_MODE        : {AMP_MODE}')
print(f'  PRELOAD_AUGMENT : {PRELOAD_AUGMENT}')
print(f'  NUM_WORKERS     : {NUM_WORKERS}')
ds_len = 80 * REPEAT
steps  = max(1, ds_len // BS)
print(f'  BS={BS:4d}  REPEAT={REPEAT:2d}  dataset_len={ds_len:4d}  steps/ep={steps}  '
      f'VRAM={peak_gb:.1f}GB  RAM={ram_mb(BS, REPEAT):.0f}MB')
print(f'  RUNS_DIR  : {RUNS_DIR}')
print(f'  EVAL_DIR  : {EVAL_DIR}')
print(f'  SEEDS     : {SEEDS}')
print(f'  NOTE      : Triplet (--loss triplet --triplet-margin 0.3); no GeoAtt, no aux')

torch.cuda.empty_cache(); gc.collect()


## 4. Cleanup (Optional)

Set `RUN_CLEANUP = True` untuk hapus runs/eval lama.

In [ ]:
import shutil

RUN_CLEANUP = False

if not RUN_CLEANUP:
    print("Skipped cleanup (RUN_CLEANUP = False)")
else:
    for d in [RUNS_DIR, EVAL_DIR]:
        if d.exists():
            shutil.rmtree(d)
            print(f"Removed: {d}")
        d.mkdir(parents=True, exist_ok=True)
    for root, dirs, files in os.walk(PROJECT_ROOT):
        for d in dirs:
            if d == '__pycache__':
                shutil.rmtree(os.path.join(root, d), ignore_errors=True)
    print("Cleanup selesai. Restart runtime sebelum lanjut.")

## 5. Gate 0 — Sanity Baseline (Geom-only LogReg, sekedar dokumentasi)

In [ ]:
!python utils/sanity_geom_only_cv.py --dataset_root dataset

## 6. Audit Temporal Gap (Dokumentasi Limitation)

In [ ]:
import datetime
TS = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
AUDIT_DIR = PROJECT_ROOT / "result_docs" / f"{TS}_v6_standard"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

!python utils/audit_temporal_gap.py \
    --dataset_root dataset \
    --output {AUDIT_DIR}/temporal_gap_audit.json

print(f'\nAudit disimpan di: {AUDIT_DIR}/temporal_gap_audit.json')

## 7. Helper Functions

In [ ]:
def run_training(seed: int) -> bool:
    out_dir = RUNS_DIR / VARIANT / f"seed_{seed}"
    ckpt = out_dir / "best.pth"
    if ckpt.exists():
        print(f"  SKIP training seed={seed} (already done)")
        return True

    amp_flag = f"--amp {AMP_MODE}" if AMP_MODE != "none" else ""
    preload_flag = "--preload-augment" if PRELOAD_AUGMENT else ""

    cmd = (
        f"python {PROJECT_ROOT / 'train.py'} "
        f"--data_dir {DATA_DIR} "
        f"--output_dir {out_dir} "
        f"--seed {seed} "
        f"--fixed_split "
        f"--frames-per-session 1 "
        f"--loss triplet --triplet-margin {TRIPLET_MARGIN} "
        f"--val-metric pair_eer --no-early-stop "
        f"--epochs {EPOCHS} --finetune_epochs {FINETUNE_EPOCHS} "
        f"--batch_size {BS} --n_points {N_POINTS} "
        f"--num_workers {NUM_WORKERS} "
        f"--repeat {REPEAT} "
        f"{amp_flag} {preload_flag} "
        f"--siamese-mode concat"
    )
    print(f"\n{'='*60}")
    print(f"START: standard (Triplet) | seed {seed} | BS={BS} N={N_POINTS} repeat={REPEAT}")
    print(f"{'='*60}")
    !{cmd}
    return ckpt.exists()


def run_eval(seed: int, split: str) -> bool:
    out_dir   = RUNS_DIR / VARIANT / f"seed_{seed}"
    ckpt      = out_dir / "best.pth"
    eval_out  = EVAL_DIR / VARIANT / f"seed_{seed}" / split
    result_file = eval_out / "results.json"

    if not ckpt.exists():
        print(f"  SKIP eval seed={seed} split={split} (checkpoint missing)")
        return False
    if result_file.exists():
        print(f"  SKIP eval seed={seed} split={split} (already done)")
        return True

    eval_out.mkdir(parents=True, exist_ok=True)
    cmd = (
        f"python {PROJECT_ROOT / 'evaluate.py'} "
        f"--data_dir {DATA_DIR} "
        f"--frames-per-session 1 "
        f"--eval-split {split} "
        f"--checkpoints \"{VARIANT}={ckpt}\" "
        f"--output_dir {eval_out} "
        f"--save_scores"
    )
    print(f"  EVAL: seed {seed} | {split}")
    !{cmd}
    return result_file.exists()

print('Helpers ready (Triplet standard)')


## Runtime Shutdown Guard

Proteksi compute: **atexit** auto-shutdown jika crash + explicit shutdown di cell terakhir.

In [ ]:
import atexit

_shutdown_triggered = False

def shutdown_colab(silent=False):
    global _shutdown_triggered
    if _shutdown_triggered:
        return
    _shutdown_triggered = True
    if not silent:
        print('\n' + '=' * 60)
        print('🔌 Shutting down Colab runtime...')
        print('=' * 60)
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        if not silent:
            print(f'  Shutdown error: {e}')
            print('  Manual: Runtime > Disconnect and delete runtime')

atexit.register(shutdown_colab, silent=True)
print('⚡ Runtime shutdown guard terdaftar (atexit)')


## Git Save Helper

Fungsi untuk commit + push hasil ke branch `colab`.

In [ ]:
import subprocess, os

def git_save(message: str, push: bool = False):
    repo = str(REPO_DIR)
    os.chdir(repo)

    find_dirs = [d for d in ['3DCNN/runs', '3DCNN/eval_results', '3DCNN/analysis']
                 if os.path.isdir(os.path.join(repo, d))]
    large = ''
    if find_dirs:
        try:
            large = subprocess.check_output(
                ['find'] + find_dirs + ['-size', '+95M', '-type', 'f'],
                text=True, stderr=subprocess.DEVNULL, cwd=repo
            ).strip()
        except subprocess.CalledProcessError:
            large = ''
    if large:
        print('⚠️ File > 95MB (skip dari git):')
        for f in large.split('\n'):
            print(f'  {f}')
            subprocess.run(['git', 'update-index', '--skip-worktree', f], capture_output=True)

    subprocess.run(['git', 'add', '-A'], cwd=repo)
    result = subprocess.run(['git', 'diff', '--cached', '--quiet'], cwd=repo)
    if result.returncode == 0:
        print('  (nothing to commit)')
        return

    subprocess.run(['git', 'commit', '-m', message], cwd=repo)
    print(f'✅ Committed: {message}')

    if push:
        ret = subprocess.run(['git', 'push', 'origin', COLAB_BRANCH],
                              cwd=repo, capture_output=True, text=True)
        if ret.returncode == 0:
            print(f'✅ Pushed ke origin/{COLAB_BRANCH}')
        else:
            print(f'❌ Push error: {ret.stderr}')

    os.chdir(str(PROJECT_ROOT))

print('git_save() ready')


## 8. Training (10 seeds × PointNet++ + Triplet batch-hard loss, margin=0.3)

In [ ]:
for seed in SEEDS:
    run_training(seed)

## 9. Test + Holdout Evaluation

In [ ]:
for seed in SEEDS:
    run_eval(seed, "test")
    run_eval(seed, "holdout")

In [ ]:
# Simpan hasil ke git (incremental)
git_save('v6.0.0 standard: training + eval complete', push=True)


## 10. Quick Summary (Smoke check sebelum compare notebook)

In [ ]:
import json
import numpy as np

print("="*60)
print(f"{'Seed':<6} {'Test EER':<12} {'Holdout EER':<12}")
print("="*60)
summary = {'test': [], 'holdout': []}

for seed in SEEDS:
    row = [f"{seed:<6}"]
    for split in ['test', 'holdout']:
        f = EVAL_DIR / VARIANT / f'seed_{seed}' / split / 'results.json'
        if f.exists():
            with open(f) as fp:
                r = json.load(fp)
            eer = r[0]['eer']
            summary[split].append(eer)
            row.append(f"{eer:.4f}     ")
        else:
            row.append("missing     ")
    print(" ".join(row))

print("\n" + "="*60)
t = summary['test']; h = summary['holdout']
if t:
    print(f"Test    EER (mean±std): {np.mean(t):.4f}±{np.std(t):.4f}")
if h:
    print(f"Holdout EER (mean±std): {np.mean(h):.4f}±{np.std(h):.4f}")
print("="*60)
print("\nLanjut ke notebook v6_standard_arcface_compare.ipynb")

## Auto-Shutdown Compute

Otomatis shutdown runtime setelah semua proses selesai.
Delay 60 detik untuk review output. Cancel: **Runtime > Interrupt execution**.

In [ ]:
import time

SHUTDOWN_DELAY = 60

print('=' * 60)
print('✅ ALL DONE — v6.0.0 Standard (Triplet) Train + Eval selesai!')
print('=' * 60)
print(f'\nRuntime akan di-shutdown dalam {SHUTDOWN_DELAY} detik.')
print('Untuk membatalkan: Runtime > Interrupt execution\n')

try:
    for remaining in range(SHUTDOWN_DELAY, 0, -10):
        print(f'  Shutdown dalam {remaining} detik...')
        time.sleep(min(10, remaining))
    shutdown_colab()
except KeyboardInterrupt:
    print('\n⚠️ Shutdown dibatalkan. Runtime tetap aktif.')
    _shutdown_triggered = False
